In [ ]:
import boto3
import lithops
import rasterio
from dataplug import CloudObject
from dataplug.formats.geospatial.cog import CloudOptimizedGeoTiff

In [ ]:
def get_s3_config():
    session = boto3.Session()
    creds = session.get_credentials().get_frozen_credentials()
    return {
        "credentials": {
            "AccessKeyId": creds.access_key,
            "SecretAccessKey": creds.secret_key,
            "SessionToken": creds.token,
        },
        "region_name": session.region_name,
    }

In [ ]:
def estimate_chunk_size_local(uri):
    s3_config = get_s3_config()
    co = CloudObject.from_s3(CloudOptimizedGeoTiff, f"s3://{uri}", s3_config=s3_config)
    co.preprocess()  # Required to access image size

    width = co.attributes.width
    height = co.attributes.height
    total_pixels = width * height

    # Heuristics
    pixels_per_chunk = 20_000_000
    best_chunk = max(1, total_pixels // pixels_per_chunk)

    pixels_per_mb = 1_000_000
    base_mb = (total_pixels // pixels_per_mb) + 512
    runtime_memory = min(max(512, (base_mb // 512) * 512), 8192)

    return {
        "uri": uri,
        "width": width,
        "height": height,
        "total_pixels": total_pixels,
        "best_chunk": best_chunk,
        "runtime_memory": runtime_memory
    }